# Plant Hydraulic Models

Interactive notebook entry point for the forward and inverse network models. The calculations and plotting functions live in `model.py` and `inverse_model.py`; this notebook imports and calls them rather than duplicating their implementations.

**Boundary-condition note:** the forward model fixes root and leaf pressures. The inverse model fixes root-tip pressure and prescribes leaf-tip flow, solving leaf pressures. These are different boundary-condition setups.

## 1. Model scope and physical assumptions

This is a **reduced-order hydraulic network model**, not a full 3D CFD simulation or a complete biological model of xylem transport. Each branch is represented as a rigid circular channel with a resistance determined by its length, radius, and the fluid viscosity.

The current model assumes:

- **Steady-state flow:** conditions do not change with time.
- **Incompressible, Newtonian water:** density and dynamic viscosity are treated as constant.
- **Rigid circular channels:** each branch has fixed length and radius.
- **Laminar, fully developed flow in each branch:** pressure loss is represented by the Hagen–Poiseuille relation.
- **Constant branch resistance:** resistance depends on the current geometry and viscosity, not on flow rate.
- **Mass conservation at internal junctions:** no water is created, stored, or lost at a junction.
- **Gravity included through elevation:** node heights are derived from the branch geometry.

The model does not yet include changing channel radius, elastic walls, transient storage, cavitation, solute transport, detailed biological processes, or an explicit active pump/source. Poiseuille resistance is not a universal pressure-loss model for every flow regime or geometry.

## 2. Governing equations

### 2.1 Branch hydraulic resistance — Hagen–Poiseuille

For steady, fully developed laminar flow through a rigid circular pipe,

$$R_i = \\frac{8\\mu L_i}{\\pi r_i^4}$$

Here, $R_i$ is hydraulic resistance (Pa·s/m³), $\\mu$ is dynamic viscosity (Pa·s), $L_i$ is branch length (m), and $r_i$ is internal radius (m). Resistance is proportional to length and to the inverse fourth power of radius, so small radius changes can strongly affect flow.

### 2.2 Hydraulic potential — pressure plus elevation

$$\\phi_i = p_i + \\rho g z_i$$

Here, $p_i$ is pressure (Pa), $\\rho$ is fluid density (kg/m³), $g$ is gravitational acceleration (m/s²), and $z_i$ is elevation (m). For a branch declared from node $a$ to node $b$,

$$Q_{a\\to b} = \\frac{\\phi_a-\\phi_b}{R_{ab}}$$

Positive flow follows the declared direction $a\\to b$; negative flow means the actual flow is reversed. Including elevation means pressure difference alone is not enough to describe the driving potential when endpoints have different heights.

### 2.3 Mass conservation at internal junctions

$$\\sum_j Q_{j\\to i}=0$$

At each internal junction, total inflow equals total outflow. The solver assembles a linear conductance system from branch resistances and applies this balance at unknown-potential nodes.

### 2.4 Viscous power

The rate of mechanical energy dissipated by viscous resistance in branch $i$ is

$$P_{\\mathrm{viscous},i}=R_iQ_i^2, \\qquad P_{\\mathrm{viscous}}=\\sum_i R_iQ_i^2$$

This contribution is non-negative regardless of flow direction.

### 2.5 Gravitational potential-energy rate

For signed flow in the declared direction from $a$ to $b$, let $\\Delta z_i=z_b-z_a$:

$$P_{\\mathrm{gravity},i}=\\rho g Q_i\\Delta z_i, \\qquad P_{\\mathrm{gravity}}=\\sum_i\\rho gQ_i\\Delta z_i$$

Upward transport contributes positively; downward transport can contribute negatively.

### 2.6 Mechanical power and energy over time

The inverse model currently reports

$$P_{\\mathrm{net}}=P_{\\mathrm{viscous}}+P_{\\mathrm{gravity}}, \\qquad E=P_{\\mathrm{net}}T$$

where $T$ is the duration for which steady power is assumed constant. Treat this as the model's current mechanical-energy accounting, not a prediction of biochemical energy use or metabolic efficiency. Independently validate the energy balance against the boundary mechanical-energy balance before relying on it for changed network configurations.

## 3. Boundary conditions: forward vs. inverse

**Forward model (`model.py`):** root-tip and leaf-tip pressures are prescribed; junction pressures and branch flows are solved.

**Inverse model (`inverse_model.py`):** root-tip pressure is fixed and a flow demand `Q_MIN` is prescribed at each leaf tip; the leaf-tip pressures and internal junction states are solved.

These are different boundary-condition setups. A passive resistance network generally cannot independently prescribe both pressure and flow at the same terminal. If you need fixed root and leaf pressures while also demanding nonzero leaf flow, the model needs an explicit active source/pump mechanism.

The default inverse demand is `Q_MIN = 1e-9 m³/s` **per leaf tip**. Changing it changes the physical scenario, not merely numerical precision.

## 4. Imports and configuration

In [ ]:
%matplotlib inline
import importlib
import matplotlib.pyplot as plt
import model
import inverse_model

# Reload after editing the Python modules so the notebook picks up changes.
importlib.reload(model)
importlib.reload(inverse_model)


## 5. Forward model — prescribed terminal pressures

In [ ]:
forward_results = model.solve_network()

print('Forward model: node pressures and hydraulic potentials')
for node in forward_results['nodes']:
    print(f"{node}: z={forward_results['nodes'][node]['z']:.4f} m, "
          f"p={forward_results['pressures'][node]:.6e} Pa, "
          f"phi={forward_results['phi'][node]:.6e} Pa")

print('\\nForward model: internal-node mass conservation')
for item in model.check_mass_conservation(
        forward_results['nodes'], forward_results['branches'], forward_results['flows']):
    if item['is_internal']:
        print(f"{item['node']}: net outflow={item['net_outflow']:.3e} m^3/s, "
              f"relative residual={item['relative_residual']:.3e}, "
              f"{'PASS' if item['passed'] else 'FAIL'}")


In [ ]:
# Forward-model plots: flow structure + two hydraulic-resistance views.
model.plot_flow_results(forward_results)
model.plot_hydraulic_resistance_network(forward_results)
model.plot_hydraulic_resistance_ranking(forward_results)
plt.show()


## 6. Inverse model — prescribed minimum flow at each leaf

In [ ]:
# Q_MIN is the default flow demand per leaf tip, defined in inverse_model.py.
q_min = inverse_model.Q_MIN
inverse_results = inverse_model.solve_for_minimum_leaf_flows(q_min)
inverse_model.print_report(inverse_results, duration=1.0)


In [ ]:
# Inverse-model plots: flow structure + two hydraulic-resistance views.
inverse_model.plot_results(inverse_results)


## 7. Experiment with leaf flow demand

Edit `q_min` below to test another demand per leaf tip. This changes the physical scenario, not just numerical precision. For a linear fixed-resistance network, branch flow scales linearly with the demand and viscous power scales quadratically when all leaf demands are scaled together.

In [ ]:
q_min_test = 2.0 * inverse_model.Q_MIN  # change this multiplier as desired
inverse_test_results = inverse_model.solve_for_minimum_leaf_flows(q_min_test)
inverse_model.print_report(inverse_test_results, duration=1.0)
inverse_model.plot_results(inverse_test_results)
